5) 대화 지식 그래프 메모리

In [ ]:
# [목적] ConversationKGMemory에서 사용할 채팅 모델을 준비하는 지식 그래프 메모리 예제
# 대화에서 인물·장소·관계 정보를 그래프로 추출할 수 있도록 모델을 생성합니다.
# 다음 셀의 지식 그래프 메모리가 이 모델을 이용해 대화 내용을 구조화합니다.
from langchain_openai import ChatOpenAI
from langchain_community.memory.kg import ConversationKGMemory
from dotenv import load_dotenv

load_dotenv()

# temperature=0은 관계 정보 추출 결과를 가능한 일관되게 만들기 위한 설정입니다.
llm = ChatOpenAI(temperature=0)

In [ ]:
# [목적] 대화에서 관계를 추출해 지식 그래프로 저장하고 관련 정보를 조회하는 예제
# ConversationKGMemory는 문답을 저장하면서 대상 간의 관계를 그래프 형태로 정리합니다.
# 질문이 들어오면 전체 대화 대신 질문과 연결된 정보만 history로 제공할 수 있습니다.
# return_messages=True는 조회 결과를 역할이 구분된 메시지 목록으로 반환합니다.
memory = ConversationKGMemory(llm=llm, return_messages=True)

memory.save_context(
    {"input": "이쪽은 Pangyo에 거주 중인 김설리씨 입니다."},
    {"output": "김설리 씨는 누구시죠?"},
)

memory.save_context(
    {"input": "김설리 씨는 우리 회사의 신입 디자이너입니다."},
    {"output": "만나서 반갑습니다."},
)

memory.load_memory_variables({"input": "김설리 씨는 누구입니까?"})

In [ ]:
# [목적] 지식 그래프에서 찾은 관련 정보만 답변에 활용하는 대화 체인을 구성하는 예제
# 프롬프트의 history 자리에 그래프 메모리의 관련 정보를 넣고, ConversationChain이 모델 호출을 연결합니다.
# 대화 전체가 아니라 관계된 사실을 바탕으로 답하도록 제한할 때 사용하는 방식입니다.
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import ConversationChain

llm = ChatOpenAI(temperature=0)

template = """The following is a friendly conversation ...
The AI ONLY uses information contained in the "Relevant Information" section ...

Relevant Information:

{history}

Conversation:
Human: {input}
AI:"""

# PromptTemplate은 history와 input 값을 문장 템플릿의 {history}, {input} 위치에 채워 넣습니다.
prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=template
)

# ConversationChain은 모델, 프롬프트, 메모리를 연결해 입력부터 응답까지 처리합니다.
conversation_with_kg = ConversationChain(
    llm=llm,
    prompt=prompt,
    memory=ConversationKGMemory(llm=llm)
)

In [ ]:
# [목적] 새 대화를 처리해 인물과 직장 관계를 지식 그래프 메모리에 저장하는 예제
# predict가 답변을 생성하는 동안 이름, 동료 관계, 직무 같은 핵심 사실도 메모리에 추출됩니다.
# 저장된 관계는 다음 질문에서 관련 정보를 찾는 근거로 사용됩니다.
conversation_with_kg.predict(
    input="My name is Teddy. Shirley is a coworker of mine, and she's a new designer at our company."
)

In [ ]:
# [목적] 지식 그래프 메모리에서 Shirley와 관련된 정보를 직접 조회하는 예제
# 체인의 memory에 질문을 전달하면 해당 인물과 연결된 요약 정보가 history로 반환됩니다.
# 이 조회 결과는 다음 모델 응답에 넣을 관련 문맥을 확인하는 데 사용합니다.
conversation_with_kg.memory.load_memory_variables({"imput": "who is Shirley?"})